# GDF Viewer for VS Code

Open this notebook in VS Code, select the `cosmos` Python kernel, and run the cells. Choose a recording from the dropdown, then rerun the cells below the picker to inspect it. Files are opened read-only with `preload=False`.

In [ ]:
from pathlib import Path
import numpy as np
import ipywidgets as widgets
import matplotlib.pyplot as plt
import mne
import pandas as pd
from IPython.display import display

search_roots = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
PROJECT_ROOT = next(
    (
        root
        for root in search_roots
        if (root / "notebooks").is_dir() and (root / "data").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Run this notebook from inside the bci_cleaning project")

data_candidates = [
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "data" / "raw" / "BCI Database",
    PROJECT_ROOT / "data" / "raw",
]
DATA_ROOT = next(
    (root for root in data_candidates if root.is_dir() and any(root.rglob("*.gdf"))),
    None,
)
if DATA_ROOT is None:
    raise FileNotFoundError("No GDF recordings were found in data/processed or data/raw")

gdf_files = sorted(DATA_ROOT.rglob("*.gdf"))
print(f"Found {len(gdf_files)} GDF recordings in {DATA_ROOT.relative_to(PROJECT_ROOT)}")

In [ ]:
gdf_picker = widgets.Dropdown(
    options=[(path.relative_to(DATA_ROOT).as_posix(), str(path)) for path in gdf_files],
    description="Recording:",
    layout=widgets.Layout(width="95%"),
    style={"description_width": "initial"},
)
display(gdf_picker)

After changing the recording above, run this cell and the remaining cells again.

In [ ]:
selected_gdf = Path(gdf_picker.value)
raw = mne.io.read_raw_gdf(selected_gdf, preload=False, verbose="ERROR")

summary = pd.Series(
    {
        "file": selected_gdf.relative_to(DATA_ROOT).as_posix(),
        "channels": raw.info["nchan"],
        "sampling_frequency_hz": raw.info["sfreq"],
        "samples": raw.n_times,
        "duration_seconds": raw.times[-1],
        "annotations": len(raw.annotations),
        "preloaded": raw.preload,
    },
    name="value",
)
summary.to_frame()

In [ ]:
channel_table = pd.DataFrame(
    {
        "channel": raw.ch_names,
        "type": raw.get_channel_types(),
    }
)
channel_table

In [ ]:
annotation_table = pd.DataFrame(
    {
        "onset_seconds": raw.annotations.onset,
        "duration_seconds": raw.annotations.duration,
        "description": raw.annotations.description,
    }
)
annotation_table


In [ ]:
%matplotlib inline

plot_start_seconds = 0
plot_duration_seconds = min(30, raw.times[-1])
raw.plot(
    start=plot_start_seconds,
    duration=plot_duration_seconds,
    n_channels=min(20, raw.info["nchan"]),
    scalings="auto",
    show=False,
    block=False,
    title=selected_gdf.name,
)
plt.show()

In [ ]:
performances_path = PROJECT_ROOT / "data" / "processed" / "Perfomances.csv"
performances = pd.read_csv(
    performances_path,
    sep=";",
    encoding="utf-8",
    skiprows=2,
    dtype="string",
)

# Normalize column names and surrounding whitespace without renaming published fields.
performances.columns = performances.columns.str.strip()
for column in performances.columns:
    performances[column] = performances[column].str.strip()

# Remove the repeated Dataset B/C headings and retain only real participant records.
valid_subject_mask = performances["SUJ_ID"].str.fullmatch(r"[ABC]\d+", na=False)
performances = performances.loc[valid_subject_mask].reset_index(drop=True)

if len(performances) != 87:
    raise ValueError(f"Expected 87 participant rows, found {len(performances)}")
if performances["SUJ_ID"].duplicated().any():
    duplicates = performances.loc[
        performances["SUJ_ID"].duplicated(keep=False), "SUJ_ID"
    ].tolist()
    raise ValueError(f"Duplicate participant identifiers: {duplicates}")

# Preserve genuine text fields; every other published field is analytical numeric data.
text_columns = ["SUJ_ID", "COMMENTS", "Manual activity TXT", "PRE_Pills_TXT"]
numeric_columns = [
    column for column in performances.columns if column not in text_columns
]
numeric_missing_tokens = {"": pd.NA, "na": pd.NA, "n/a": pd.NA, "nan": pd.NA}

for column in numeric_columns:
    normalized = (
        performances[column]
        .str.lower()
        .replace(numeric_missing_tokens)
        .str.replace(",", ".", regex=False)
    )
    converted = pd.to_numeric(normalized, errors="coerce")
    invalid_mask = normalized.notna() & converted.isna()
    if invalid_mask.any():
        invalid_values = sorted(normalized.loc[invalid_mask].unique().tolist())
        raise ValueError(f"Unexpected values in {column}: {invalid_values}")
    performances[column] = converted

for column in text_columns[1:]:
    performances[column] = performances[column].replace("", pd.NA)

# Validate the strongest documented codes and ranges without deleting missing records.
for column in ["SUJ_gender", "EXP_gender"]:
    invalid_codes = performances.loc[
        performances[column].notna() & ~performances[column].isin([1, 2]), column
    ]
    if not invalid_codes.empty:
        raise ValueError(f"Invalid {column} codes: {sorted(invalid_codes.unique())}")

performance_columns = ["Perf_RUN_3", "Perf_RUN_4", "Perf_RUN_5", "Perf_RUN_6"]
for column in performance_columns:
    invalid_scores = performances.loc[
        performances[column].notna() & ~performances[column].between(0, 100), column
    ]
    if not invalid_scores.empty:
        raise ValueError(f"Out-of-range values in {column}: {invalid_scores.tolist()}")

if not performances["Birth_year"].dropna().between(1900, 2100).all():
    raise ValueError("Birth_year contains an implausible value")

cleaning_summary = pd.DataFrame(
    {
        "dtype": performances.dtypes.astype(str),
        "missing_count": performances.isna().sum(),
        "missing_percent": performances.isna().mean().mul(100).round(1),
    }
)
print(f"Cleaned dataset: {len(performances)} participants × {performances.shape[1]} columns")
display(performances.head(), cleaning_summary)

In [ ]:
gender_counts = (
    performances["SUJ_gender"]
    .value_counts()
    .reindex([1, 2], fill_value=0)
    .astype(int)
)

ax = gender_counts.plot.bar(
    color=["steelblue", "coral"],
    figsize=(7, 5),
    rot=0,
)
ax.set_title("Participant Gender Distribution")
ax.set_xlabel("SUJ_gender")
ax.set_ylabel("Number of Participants")
ax.set_xticklabels(["1", "2"])
ax.bar_label(ax.containers[0])
plt.tight_layout()
plt.show()

In [ ]:
birth = performances["Birth_year"].astype(int)

plt.figure(figsize=(8, 5))
plt.hist(birth, bins=25, color="skyblue", edgecolor="black")
plt.title("Distribution of Participant Birth Years")
plt.xlabel("Birth Year")
plt.ylabel("Number of Participants")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns

sns.kdeplot(birth, fill=True, color="blue", alpha=0.5)
plt.title("KDE Plot with Seaborn")
plt.xlabel("Values")
plt.ylabel("Density")
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 2, figsize = (10,6))
axes[0,0].hist(performances["Perf_RUN_3"], bins = 25,range=(0, 101),color = "skyblue", edgecolor = "black")
axes[0,0].set_xlabel("Percentage of Successful Runs")
axes[0,1].hist(performances["Perf_RUN_4"], bins = 25,range=(0, 101), color = "skyblue", edgecolor = "black")

axes[1,0].hist(performances["Perf_RUN_5"], bins = 25,range=(0, 101), color = "skyblue", edgecolor = "black")

axes[1,1].hist(performances["Perf_RUN_6"], bins = 25,range=(0, 101), color = "skyblue", edgecolor = "black")
plt.tight_layout()
plt.show()
'''
df = pd.DataFrame({
    'Year': [2019, 2020, 2021, 2022],
    'Sales_A': [120, 135, 150, 160],
    'Sales_B': [80, 95, 100, 120],
    'Profit_A': [30, 32, 36, 38],
    'Profit_B': [18, 20, 22, 26]
})

fig, axs = plt.subplots(2, 2, figsize=(10, 6), sharex=True, sharey=True)

# [0, 0] Sales of A
axs[0, 0].bar(df['Year'], df['Sales_A'], color='steelblue')
axs[0, 0].set_title("Sales - Product A")

# [0, 1] Sales of B
axs[0, 1].bar(df['Year'], df['Sales_B'], color='salmon')
axs[0, 1].set_title("Sales - Product B")

# [1, 0] Profit of A
axs[1, 0].plot(df['Year'], df['Profit_A'], color='seagreen', marker='o')
axs[1, 0].set_title("Profit - Product A")

# [1, 1] Profit of B
axs[1, 1].plot(df['Year'], df['Profit_B'], color='orange', marker='o')
axs[1, 1].set_title("Profit - Product B")


for ax in axs[1, :]:
    ax.set_xticks(df['Year'])
    ax.set_xticklabels(df['Year'])

for ax in axs[:, 0]:   
    ax.set_ylabel("Amount (k$)")

plt.tight_layout()
plt.show()
'''

In [ ]:
# Group the 16PF5 primary factors into broader subject-level categories.
# A trailing minus sign in the supplied mapping means reverse scoring (11 - score)
# because the primary-factor values use a 1-to-10 scale.
bucket_factors = {
    "Extraversion / Assertiveness": {
        "positive": ["A", "E", "F", "H"],
        "reverse": ["N", "Q2"],
    },
    "Anxiety / Emotional Stability": {
        "positive": ["L", "O", "Q4"],
        "reverse": ["C_"],
    },
    "Openness / Tough-Mindedness": {
        "positive": ["M", "Q1"],
        "reverse": ["A", "I"],
    },
    "Self-Control / Conscientiousness": {
        "positive": ["G", "Q3"],
        "reverse": ["M", "Q1"],
    },
}

# Keep only actual participants; the source CSV contains repeated header rows.
subject_mask = performances["SUJ_ID"].astype("string").str.fullmatch(r"[ABC]\d+")
subject_16pf = performances.loc[subject_mask].copy()

primary_factors = sorted(
    {
        factor
        for mapping in bucket_factors.values()
        for direction in ("positive", "reverse")
        for factor in mapping[direction]
    }
)
subject_16pf[primary_factors] = subject_16pf[primary_factors].apply(
    pd.to_numeric, errors="coerce"
)

category_scores = pd.DataFrame({"SUJ_ID": subject_16pf["SUJ_ID"]}).reset_index(drop=True)
for category, mapping in bucket_factors.items():
    components = [subject_16pf[factor] for factor in mapping["positive"]]
    components.extend(11 - subject_16pf[factor] for factor in mapping["reverse"])
    category_scores[category] = pd.concat(components, axis=1).mean(axis=1).reset_index(drop=True)

display(category_scores.head())

fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharey=True)
for ax, category in zip(axes.flat, bucket_factors):
    sns.boxplot(y=category_scores[category], ax=ax, color="skyblue", width=0.45)
    sns.stripplot(y=category_scores[category], ax=ax, color="black", alpha=0.45, size=3)
    ax.set_title(category)
    ax.set_xlabel("")
    ax.set_ylabel("Mean reverse-adjusted 16PF5 score")
    ax.set_ylim(1, 10)
    ax.grid(axis="y", alpha=0.25)

fig.suptitle("Broader 16PF5 Categories Across Participants", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Compare the published broad 16PF5 factor columns with mean online BCI performance.
run_columns = ["Perf_RUN_3", "Perf_RUN_4", "Perf_RUN_5", "Perf_RUN_6"]
broad_factor_labels = {
    "EX": "Extraversion",
    "AX": "Anxiety",
    "TM": "Tough-Mindedness",
    "IN": "Independence",
    "SC": "Self-Control",
}

personality_performance = performances[
    ["SUJ_ID", *run_columns, *broad_factor_labels]
].copy()
numeric_columns = [*run_columns, *broad_factor_labels]
for column in numeric_columns:
    personality_performance[column] = pd.to_numeric(
        personality_performance[column]
        .astype("string")
        .str.replace(",", ".", regex=False),
        errors="coerce",
    )
personality_performance["Mean_Performance"] = personality_performance[
    run_columns
].mean(axis=1, skipna=True)
display(
    personality_performance[
        ["SUJ_ID", *broad_factor_labels, "Mean_Performance"]
    ].head()
)

fig, axes = plt.subplots(2, 3, figsize=(14, 8), sharey=True)
for ax, (factor, label) in zip(axes.flat, broad_factor_labels.items()):
    plot_data = personality_performance[[factor, "Mean_Performance"]].dropna()
    ax.scatter(
        plot_data[factor],
        plot_data["Mean_Performance"],
        color="steelblue",
        edgecolor="black",
        alpha=0.75,
    )
    ax.set_title(f"{factor}: {label}")
    ax.set_xlabel(f"{factor} score")
    ax.set_ylabel("Mean accuracy across Runs 3–6 (%)")
    ax.set_ylim(40, 100)
    ax.grid(alpha=0.25)

axes.flat[-1].set_visible(False)
fig.suptitle("16PF5 Broad Factors vs. Mean Online BCI Performance", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
from itertools import combinations

# combinations() creates each unordered pair exactly once: 5 choose 2 = 10 plots.
broad_factors = ["EX", "AX", "TM", "IN", "SC"]
factor_pairs = list(combinations(broad_factors, 2))

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for ax, (x_factor, y_factor) in zip(axes.flat, factor_pairs):
    plot_data = performances[[x_factor, y_factor]].dropna()
    correlation = plot_data[x_factor].corr(plot_data[y_factor])
    ax.scatter(
        plot_data[x_factor],
        plot_data[y_factor],
        color="mediumpurple",
        edgecolor="black",
        alpha=0.7,
    )
    if len(plot_data) >= 2 and plot_data[x_factor].nunique() > 1:
        x_values = plot_data[x_factor].astype(float).to_numpy()
        y_values = plot_data[y_factor].astype(float).to_numpy()
        slope, intercept = np.polyfit(x_values, y_values, 1)
        fit_x = np.linspace(x_values.min(), x_values.max(), 100)
        ax.plot(
            fit_x,
            slope * fit_x + intercept,
            color="darkorange",
            linewidth=2,
            label="Best fit",
        )
        ax.legend()
    ax.set_title(f"{x_factor} vs. {y_factor}\nr = {correlation:.2f}")
    ax.set_xlabel(f"{x_factor} score")
    ax.set_ylabel(f"{y_factor} score")
    ax.grid(alpha=0.25)

fig.suptitle("Unique Pairwise Comparisons of 16PF5 Broad Factors", fontsize=16)
plt.tight_layout()
plt.show()

print(f"Created {len(factor_pairs)} unique scatterplots: {factor_pairs}")

In [ ]:
# Compare each Index of Learning Style score with mean online BCI performance.
learning_style_columns = [
    "active", "reflexive", "sensory", "intuitive",
    "visual", "verbal", "sequential", "global",
]
run_columns = ["Perf_RUN_3", "Perf_RUN_4", "Perf_RUN_5", "Perf_RUN_6"]

learning_style_performance = performances[
    ["SUJ_ID", *learning_style_columns, *run_columns]
].copy()
learning_style_performance["Mean_Performance"] = learning_style_performance[
    run_columns
].mean(axis=1, skipna=True)

fig, axes = plt.subplots(2, 4, figsize=(18, 9), sharey=True)
for ax, style in zip(axes.flat, learning_style_columns):
    plot_data = learning_style_performance[[style, "Mean_Performance"]].dropna()
    correlation = plot_data[style].corr(plot_data["Mean_Performance"])
    ax.scatter(
        plot_data[style],
        plot_data["Mean_Performance"],
        color="teal",
        edgecolor="black",
        alpha=0.75,
    )
    if len(plot_data) >= 2 and plot_data[style].nunique() > 1:
        x_values = plot_data[style].astype(float).to_numpy()
        y_values = plot_data["Mean_Performance"].astype(float).to_numpy()
        slope, intercept = np.polyfit(x_values, y_values, 1)
        fit_x = np.linspace(x_values.min(), x_values.max(), 100)
        ax.plot(fit_x, slope * fit_x + intercept, color="darkorange", linewidth=2)
    ax.set_title(f"{style.capitalize()} vs. Performance\nr = {correlation:.2f}")
    ax.set_xlabel(f"{style.capitalize()} score")
    ax.set_ylabel("Mean accuracy across Runs 3–6 (%)")
    ax.set_ylim(40, 100)
    ax.grid(alpha=0.25)

fig.suptitle("Index of Learning Styles vs. Mean Online BCI Performance", fontsize=16)
plt.tight_layout()
plt.show()

display(learning_style_performance.head())

In [ ]:
# Compare the mental-rotation score with each online BCI performance run.
run_columns = ["Perf_RUN_3", "Perf_RUN_4", "Perf_RUN_5", "Perf_RUN_6"]
score_performance = performances[["SUJ_ID", "score", *run_columns]].copy()

for column in ["score", *run_columns]:
    score_performance[column] = pd.to_numeric(
        score_performance[column], errors="coerce"
    )

fig, axes = plt.subplots(2, 2, figsize=(12, 9), sharex=True, sharey=True)
for ax, run in zip(axes.flat, run_columns):
    plot_data = score_performance[["score", run]].dropna()
    correlation = plot_data["score"].corr(plot_data[run])
    ax.scatter(
        plot_data["score"],
        plot_data[run],
        color="steelblue",
        edgecolor="black",
        alpha=0.75,
    )

    if len(plot_data) >= 2 and plot_data["score"].nunique() > 1:
        x_values = plot_data["score"].to_numpy(dtype=float)
        y_values = plot_data[run].to_numpy(dtype=float)
        slope, intercept = np.polyfit(x_values, y_values, 1)
        fit_x = np.linspace(x_values.min(), x_values.max(), 100)
        ax.plot(
            fit_x,
            slope * fit_x + intercept,
            color="darkorange",
            linewidth=2,
            label="Best fit",
        )
        ax.legend()

    ax.set_title(f"Mental-Rotation Score vs. {run}\nr = {correlation:.2f}")
    ax.set_xlabel("Mental-rotation score")
    ax.set_ylabel("Successful runs (%)")
    ax.set_ylim(0, 100)
    ax.grid(alpha=0.25)

fig.suptitle("Mental-Rotation Score vs. Online BCI Performance", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Plot performance distributions for participants whose SUJ_gender code is 1.
run_columns = ["Perf_RUN_3", "Perf_RUN_4", "Perf_RUN_5", "Perf_RUN_6"]
gender_one = performances.loc[performances["SUJ_gender"] == 1].copy()

gender_one_run_averages = {}
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
for ax, run in zip(axes.flat, run_columns):
    run_values = pd.to_numeric(gender_one[run], errors="coerce").dropna()
    average = run_values.mean()
    gender_one_run_averages[run] = average
    ax.hist(
        run_values,
        bins=20,
        range=(0, 100),
        color="skyblue",
        edgecolor="black",
    )
    ax.axvline(
        average,
        color="darkorange",
        linestyle="--",
        linewidth=2,
        label=f"Average = {average:.2f}%",
    )
    ax.set_title(f"{run} (n={len(run_values)}, average={average:.2f}%)")
    ax.set_xlabel("Successful runs (%)")
    ax.set_ylabel("Number of participants")
    ax.grid(axis="y", alpha=0.25)
    ax.legend()

fig.suptitle("Performance Runs for Participants with SUJ_gender = 1", fontsize=16)
plt.tight_layout()
plt.show()

print(f"Participants with SUJ_gender = 1: {len(gender_one)}")
display(
    pd.Series(
        gender_one_run_averages,
        name="Average successful run percentage",
    ).to_frame()
)

In [ ]:
# Plot performance distributions for participants whose SUJ_gender code is 2.
run_columns = ["Perf_RUN_3", "Perf_RUN_4", "Perf_RUN_5", "Perf_RUN_6"]
gender_two = performances.loc[performances["SUJ_gender"] == 2].copy()

gender_two_run_averages = {}
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
for ax, run in zip(axes.flat, run_columns):
    run_values = pd.to_numeric(gender_two[run], errors="coerce").dropna()
    average = run_values.mean()
    gender_two_run_averages[run] = average
    ax.hist(
        run_values,
        bins=20,
        range=(0, 100),
        color="coral",
        edgecolor="black",
    )
    ax.axvline(
        average,
        color="darkred",
        linestyle="--",
        linewidth=2,
        label=f"Average = {average:.2f}%",
    )
    ax.set_title(f"{run} (n={len(run_values)}, average={average:.2f}%)")
    ax.set_xlabel("Successful runs (%)")
    ax.set_ylabel("Number of participants")
    ax.grid(axis="y", alpha=0.25)
    ax.legend()

fig.suptitle("Performance Runs for Participants with SUJ_gender = 2", fontsize=16)
plt.tight_layout()
plt.show()

print(f"Participants with SUJ_gender = 2: {len(gender_two)}")
display(
    pd.Series(
        gender_two_run_averages,
        name="Average successful run percentage",
    ).to_frame()
)

In [ ]:
# Compare numeric PRE-session measures (CSV columns 22–35) with mean BCI performance.
pre_columns = [
    "PRE_Mood",
    "PRE_Mindfulness",
    "PRE_Motivation",
    "PRE_Hours_sleep_last_night",
    "PRE_Usual_sleep",
    "PRE_Level_of_alertness",
    "PRE_Stimulant_doses_12h",
    "PRE_Stimulant_doses_2h",
    "PRE_Stim_normal",
    "PRE_Tabacco",
    "PRE_Tabacco_normal",
    "PRE_Alcohol",
    "PRE_Last_meal",
    "PRE_Last_pills",
]
run_columns = ["Perf_RUN_3", "Perf_RUN_4", "Perf_RUN_5", "Perf_RUN_6"]

pre_performance = performances[
    ["SUJ_ID", *pre_columns, *run_columns]
].copy()
for column in [*pre_columns, *run_columns]:
    pre_performance[column] = pd.to_numeric(
        pre_performance[column], errors="coerce"
    )
pre_performance["Mean_Performance"] = pre_performance[run_columns].mean(
    axis=1, skipna=True
)

fig, axes = plt.subplots(3, 5, figsize=(20, 12), sharey=True)
for ax, pre_column in zip(axes.flat, pre_columns):
    plot_data = pre_performance[[pre_column, "Mean_Performance"]].dropna()
    correlation = plot_data[pre_column].corr(plot_data["Mean_Performance"])
    bar_data = (
        plot_data.groupby(pre_column, as_index=False)["Mean_Performance"]
        .mean()
        .sort_values(pre_column)
    )
    unique_x = bar_data[pre_column].to_numpy(dtype=float)
    x_gaps = np.diff(unique_x)
    bar_width = 0.8 * x_gaps[x_gaps > 0].min() if (x_gaps > 0).any() else 0.8
    ax.bar(
        bar_data[pre_column],
        bar_data["Mean_Performance"],
        width=bar_width,
        color="royalblue",
        edgecolor="black",
        alpha=0.7,
    )

    if len(plot_data) >= 2 and plot_data[pre_column].nunique() > 1:
        x_values = plot_data[pre_column].to_numpy(dtype=float)
        y_values = plot_data["Mean_Performance"].to_numpy(dtype=float)
        slope, intercept = np.polyfit(x_values, y_values, 1)
        fit_x = np.linspace(x_values.min(), x_values.max(), 100)
        ax.plot(
            fit_x,
            slope * fit_x + intercept,
            color="darkorange",
            linewidth=2,
            label="Best fit",
        )
        ax.legend()

    ax.set_title(f"{pre_column}\nr = {correlation:.2f}, n = {len(plot_data)}")
    ax.set_xlabel(pre_column.replace("PRE_", "").replace("_", " "))
    ax.set_ylabel("Mean accuracy across Runs 3–6 (%)")
    ax.set_ylim(0, 100)
    ax.grid(alpha=0.25)

# Fourteen measures use fourteen of the fifteen subplot positions.
axes.flat[-1].set_visible(False)
fig.suptitle(
    "PRE-Session Measures vs. Mean Online BCI Performance (Bar Graphs)",
    fontsize=16,
)
plt.tight_layout()
plt.show()

In [ ]:
# Plot mean performance for each run with 95% confidence intervals.
run_columns = ["Perf_RUN_3", "Perf_RUN_4", "Perf_RUN_5", "Perf_RUN_6"]
run_labels = ["Run 3", "Run 4", "Run 5", "Run 6"]

run_statistics = []
for run, label in zip(run_columns, run_labels):
    values = pd.to_numeric(performances[run], errors="coerce").dropna()
    mean = values.mean()
    standard_error = values.sem()
    margin_of_error = 1.96 * standard_error
    run_statistics.append(
        {
            "Run": label,
            "n": len(values),
            "Mean_Performance": mean,
            "CI_95_Lower": mean - margin_of_error,
            "CI_95_Upper": mean + margin_of_error,
            "CI_95_Margin": margin_of_error,
        }
    )

run_summary = pd.DataFrame(run_statistics)

fig, ax = plt.subplots(figsize=(9, 6))
ax.errorbar(
    run_summary["Run"],
    run_summary["Mean_Performance"],
    yerr=run_summary["CI_95_Margin"],
    fmt="o-",
    color="royalblue",
    ecolor="darkorange",
    markeredgecolor="black",
    markersize=8,
    linewidth=2,
    elinewidth=2,
    capsize=6,
)

for index, row in run_summary.iterrows():
    ax.annotate(
        f"{row['Mean_Performance']:.2f}%",
        (index, row["Mean_Performance"]),
        xytext=(0, 10),
        textcoords="offset points",
        ha="center",
    )

ax.set_title("Mean Performance per Run with 95% Confidence Intervals")
ax.set_xlabel("Performance run")
ax.set_ylabel("Mean successful runs (%)")
ax.set_ylim(0, 100)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

display(
    run_summary[
        ["Run", "n", "Mean_Performance", "CI_95_Lower", "CI_95_Upper"]
    ].round(2)
)

In [ ]:
# Examine aggregated BCI accuracy for left-handed participants.
# Dataset coding: Laterality answered — Right=1, Left=2, Ambidextrous=3.
run_columns = ["Perf_RUN_3", "Perf_RUN_4", "Perf_RUN_5", "Perf_RUN_6"]
laterality_performance = performances[
    ["SUJ_ID", "Laterality answered", *run_columns]
].copy()
for column in ["Laterality answered", *run_columns]:
    laterality_performance[column] = pd.to_numeric(
        laterality_performance[column], errors="coerce"
    )
laterality_performance["Mean_Performance"] = laterality_performance[
    run_columns
].mean(axis=1, skipna=True)

left_handed = laterality_performance.loc[
    laterality_performance["Laterality answered"] == 2,
    ["SUJ_ID", "Laterality answered", "Mean_Performance"],
].dropna(subset=["Mean_Performance"])

if left_handed.empty:
    raise ValueError("No left-handed participants (Laterality answered = 2) were found")

# Jitter reveals participants that would otherwise overlap at the single x-value of 2.
rng = np.random.default_rng(42)
jittered_laterality = left_handed["Laterality answered"] + rng.uniform(
    -0.045, 0.045, len(left_handed)
)
left_mean = left_handed["Mean_Performance"].mean()

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(
    jittered_laterality,
    left_handed["Mean_Performance"],
    color="mediumpurple",
    edgecolor="black",
    alpha=0.8,
    s=70,
    label="Participant",
)
ax.hlines(
    left_mean,
    xmin=1.92,
    xmax=2.08,
    color="darkorange",
    linewidth=3,
    label=f"Left-handed mean = {left_mean:.2f}%",
)
ax.set_xticks([2], ["Left-handed (2)"])
ax.set_xlim(1.85, 2.15)
ax.set_ylim(0, 100)
ax.set_xlabel("Laterality answered")
ax.set_ylabel("Mean accuracy across Runs 3–6 (%)")
ax.set_title(
    f"Aggregated Performance for Left-Handed Participants (n={len(left_handed)})"
)
ax.grid(axis="y", alpha=0.25)
ax.legend()
plt.tight_layout()
plt.show()

display(left_handed.round(2))

In [ ]:
# Save the fully cleaned participant table without overwriting the source CSV.
cleaned_performances_path = (
    PROJECT_ROOT / "data" / "processed" / "Perfomances_cleaned.csv"
)
performances.to_csv(
    cleaned_performances_path,
    sep=";",
    index=False,
    encoding="utf-8",
)

# Verify that the saved file has the expected participant and column counts.
saved_performances = pd.read_csv(cleaned_performances_path, sep=";")
if saved_performances.shape != performances.shape:
    raise ValueError(
        f"Saved shape {saved_performances.shape} does not match "
        f"cleaned shape {performances.shape}"
    )

print(
    f"Saved {saved_performances.shape[0]} participants × "
    f"{saved_performances.shape[1]} columns to {cleaned_performances_path}"
)